# C.3 多試幾次能找到正確答案嗎？

# C：推理步驟、驗證與測試時算力

前置：下一字生成、SFT與標準答案驗證。像解數學題：寫出步驟、多試幾種辦法、檢查答案，是三個不同動作。真實推理質量需要trained模型；默認例子用可驗證的數字，避免把長文字當思考保證。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：多試幾次，還要知道怎麼選**

候選包含正解，不代表最後答案已經選中。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/reasoning.svg")))

**先想一想：**出現一篇正確答案，就代表最後交出去那篇正確嗎？

多抽幾個候選更可能包含正確答案，但會多花生成成本。候選含正解叫coverage，不是系統已經選中了正解。

**把直覺寫成數學：**若獨立單次成功率p，至少一次正確約1−(1−p)^N；真實採樣可能高度相關。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
p = 0.3
for n in (1, 2, 4, 8):
    print(n, "獨立假設coverage", 1 - (1 - p) ** n)
candidates = [3, 4, 3, 5]
print("候選包含正確答案4", 4 in candidates)

**如何讀結果：**這些是公式與預錄候選示例，實際pass@k需真實採樣並保留生成設置。

**只改一件事：**只讓候選都重複同一答案，說明獨立假設不成立。
